In [1]:
%pylab inline
import pandas as pd
import numpy as np

import os
import sys

%pylab is deprecated, use %matplotlib inline and import the required libraries.
Populating the interactive namespace from numpy and matplotlib


In [2]:
# Указываем переменные окружения
os.environ["SPARK_HOME"] = "/opt/homebrew/opt/apache-spark/libexec"
os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home"

# Добавим Spark Python API в sys.path
sys.path.append("/opt/homebrew/opt/apache-spark/libexec/python")
sys.path.append("/opt/homebrew/opt/apache-spark/libexec/python/lib/py4j-0.10.9.7-src.zip") 

In [3]:
import findspark
findspark.init()

In [4]:
from pyspark.sql import SparkSession

spark = (
    SparkSession
        .builder
        .appName("OTUS")
        .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/09/02 18:56:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/09/02 18:56:02 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/09/02 18:56:02 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


In [5]:
lectures_df = spark.read.csv("../data/lectures.csv", inferSchema=True, header=True)
question_df = spark.read.csv("../data/questions.csv", inferSchema=True, header=True)

In [6]:
lectures_df.show(2)

+----------+---+----+-------+
|lecture_id|tag|part|type_of|
+----------+---+----+-------+
|        89|159|   5|concept|
|       100| 70|   1|concept|
+----------+---+----+-------+
only showing top 2 rows


In [7]:
question_df.show(2)

+-----------+---------+--------------+----+-------------+
|question_id|bundle_id|correct_answer|part|         tags|
+-----------+---------+--------------+----+-------------+
|          0|        0|             0|   1|51 131 162 38|
|          1|        1|             1|   1|    131 36 81|
+-----------+---------+--------------+----+-------------+
only showing top 2 rows


In [8]:
lectures_df.printSchema()

root
 |-- lecture_id: integer (nullable = true)
 |-- tag: integer (nullable = true)
 |-- part: integer (nullable = true)
 |-- type_of: string (nullable = true)



In [9]:
question_df.printSchema()

root
 |-- question_id: integer (nullable = true)
 |-- bundle_id: integer (nullable = true)
 |-- correct_answer: integer (nullable = true)
 |-- part: integer (nullable = true)
 |-- tags: string (nullable = true)



поле tags у нас разного типа

In [ ]:
from pyspark.sql.functions import col, split, explode, trim

In [11]:
questions_df_exploded = question_df \
    .withColumn("tag_str", explode(split(trim(col("tags")), " "))) \
    .withColumn("tag_str", trim(col("tag_str"))) \
    .filter(col("tag_str").rlike("^[0-9]+$")) \
    .withColumn("tag", col("tag_str").cast("int"))

questions_df_exploded.show()

+-----------+---------+--------------+----+--------------+-------+---+
|question_id|bundle_id|correct_answer|part|          tags|tag_str|tag|
+-----------+---------+--------------+----+--------------+-------+---+
|          0|        0|             0|   1| 51 131 162 38|     51| 51|
|          0|        0|             0|   1| 51 131 162 38|    131|131|
|          0|        0|             0|   1| 51 131 162 38|    162|162|
|          0|        0|             0|   1| 51 131 162 38|     38| 38|
|          1|        1|             1|   1|     131 36 81|    131|131|
|          1|        1|             1|   1|     131 36 81|     36| 36|
|          1|        1|             1|   1|     131 36 81|     81| 81|
|          2|        2|             0|   1|131 101 162 92|    131|131|
|          2|        2|             0|   1|131 101 162 92|    101|101|
|          2|        2|             0|   1|131 101 162 92|    162|162|
|          2|        2|             0|   1|131 101 162 92|     92| 92|
|     

In [12]:
joined_df = questions_df_exploded.join(lectures_df, on="tag", how="inner")

In [14]:
joined_df.show(10)

+---+-----------+---------+--------------+----+--------------+-------+----------+----+----------------+
|tag|question_id|bundle_id|correct_answer|part|          tags|tag_str|lecture_id|part|         type_of|
+---+-----------+---------+--------------+----+--------------+-------+----------+----+----------------+
| 51|          0|        0|             0|   1| 51 131 162 38|     51|     19147|   1|solving question|
| 51|          0|        0|             0|   1| 51 131 162 38|     51|     17329|   1|         concept|
| 36|          1|        1|             1|   1|     131 36 81|     36|     22938|   1|solving question|
| 36|          1|        1|             1|   1|     131 36 81|     36|     19411|   1|         concept|
| 36|          1|        1|             1|   1|     131 36 81|     36|     13704|   1|solving question|
| 36|          1|        1|             1|   1|     131 36 81|     36|      7785|   1|solving question|
|101|          2|        2|             0|   1|131 101 162 92|  